# IngredientMassEstimation V3 — gramos condicionales + BCE + RGB/depth 320

Nueva variante, sin modificar V2. Usa el split **provisional agrupado** aceptado; no garantiza independencia de todos los escaneos.
ConvNeXt Small (stem/etapa 1 congelados), etapas 2–4 ajustables, dos capas Query2Label, MLP 768→384→192.
Todas las operaciones de entrenamiento, calibración, inferencia y generación de depth están desactivadas inicialmente.
No ejecutar “Run All” esperando que entrene: hay que habilitar explícitamente cada operación.


## 1. Montar Google Drive

Usar la cuenta que tiene acceso a la carpeta. Si está compartida, crear un acceso directo en Mi unidad o adaptar DRIVE en el punto 3.


In [ ]:
from pathlib import Path
import os, sys, json, subprocess, shutil, zipfile, hashlib
from IPython.display import display, HTML
from google.colab import drive
drive.mount("/content/drive")


## 2. Descargar la rama V3 publicada

Publicar primero el código y configurar BRANCH. Para repositorio privado: PRIVATE_REPO=True y secreto GITHUB_TOKEN; no se imprime ni persiste el token en la URL.


In [ ]:
URL = "https://github.com/marcogloazzo/IngredientMassEstimation.git"
BRANCH = "v3-conditional-mass"
PRIVATE_REPO = False
REPO = Path("/content/nutrition-code-v3")

def git_private(command):
    environment = os.environ.copy()
    try:
        if PRIVATE_REPO:
            from google.colab import userdata
            import base64
            token = userdata.get("GITHUB_TOKEN")
            authorization = base64.b64encode(f"x-access-token:{token}".encode()).decode()
            environment.update({
                "GIT_CONFIG_COUNT": "1", "GIT_CONFIG_KEY_0": "http.extraHeader",
                "GIT_CONFIG_VALUE_0": f"Authorization: Basic {authorization}",
            })
            del token, authorization
        result = subprocess.run(command, env=environment, capture_output=True, text=True)
        if result.returncode:
            raise RuntimeError("Git falló: revisá URL, rama publicada y permisos del token. No se imprime la credencial.")
        return result.stdout.strip()
    finally:
        environment.clear()

if (REPO / ".git").is_dir():
    current = git_private(["git", "-C", str(REPO), "branch", "--show-current"])
    assert current == BRANCH, f"Esta carpeta tiene otra rama: {current}. Usá una carpeta nueva."
    git_private(["git", "-C", str(REPO), "pull", "--ff-only", "origin", BRANCH])
else:
    git_private(["git", "clone", "--branch", BRANCH, "--single-branch", URL, str(REPO)])

PROJECT = next((p for p in (REPO, REPO / "IngredientMassEstimation")
                if (p / "config/query2label_100_v3.json").is_file()), None)
assert PROJECT is not None, "Falta publicar el código V3."
os.chdir(PROJECT)
print("Proyecto:", PROJECT)
print("Rama:", git_private(["git", "-C", str(REPO), "branch", "--show-current"]))

print("Commit:", git_private(["git", "-C", str(REPO), "rev-parse", "HEAD"]))


## 3. Dependencias y rutas exclusivas

V3 escribe solo en DRIVE/v3/. V2 permanece en DRIVE/v2/.
REPORT_TAG debe ser nuevo para una evaluación nueva: los reportes V3 no se sobrescriben.


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements-v3.txt", "pandas", "matplotlib"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".", "--no-deps"], check=True)
import torch, numpy as np
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo:", DEVICE, "| Torch:", torch.__version__)

DRIVE = Path("/content/drive/MyDrive/IngredientMassEstimation")
V2 = DRIVE / "v2"
V3 = DRIVE / "v3"
assert V3.resolve() != V2.resolve()
V3.mkdir(parents=True, exist_ok=True)
CONFIG = PROJECT / "config/query2label_100_v3.json"
DATA_PARENT = Path("/content/nutrition5k_v3")
LABELS_PARENT = Path("/content/v3_prepared")
PROCESSED = LABELS_PARENT / "processed_100_grouped_provisional"
DEPTH_REPO = Path("/content/depth-anything-v3")
DEPTH_WEIGHTS = V3 / "weights/depth_anything_v2_vits.pth"
DEPTH_CACHE = Path("/content/depth_v3_320")
DEPTH_ARCHIVE = V3 / "depth_v3_320.zip"
RUN = V3 / "runs/query2label_100_v3_bce"
REPORT_TAG = "run_01"
VALIDATION = V3 / "reports" / REPORT_TAG / "validation"
TEST = V3 / "reports" / REPORT_TAG / "test"
CHECKPOINT = VALIDATION / "calibrated.pt"
PREDICTIONS = V3 / "predictions" / REPORT_TAG

def run_stream(arguments):
    command = [str(arg) for arg in arguments]
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
        returncode = process.wait()
    except BaseException:
        process.terminate(); process.wait(); raise
    finally:
        process.stdout.close()
    if returncode:
        raise subprocess.CalledProcessError(returncode, command)


## 4. Restaurar RGB y etiquetas agrupadas congeladas

Drive debe contener nutrition5k_rgb.zip y v3/data/processed_100_grouped_provisional.zip.
**NO ejecutar prepare_nutrition5k.py**: volvería a generar el split individual.
El ZIP de etiquetas fue generado localmente y contiene vocabulario/perfiles del nuevo train.


In [ ]:
from pathlib import PurePosixPath
import stat

def extract_zip_safe(archive, destination):
    destination = Path(destination).resolve()
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as source:
        checked = []
        for info in source.infolist():
            name = info.filename.replace("\\", "/")
            relative = PurePosixPath(name)
            if relative.is_absolute() or not relative.parts or ":" in relative.parts[0]:
                raise ValueError(f"Nombre inseguro en ZIP: {name}")
            target = (destination / relative).resolve()
            if not target.is_relative_to(destination):
                raise ValueError(f"Ruta insegura en ZIP: {name}")
            if stat.S_ISLNK(info.external_attr >> 16):
                raise ValueError(f"No se aceptan enlaces en ZIP: {name}")
            checked.append((info, target, name.endswith("/")))
        for info, target, is_directory in checked:
            if is_directory:
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                with source.open(info) as incoming, target.open("wb") as outgoing:
                    shutil.copyfileobj(incoming, outgoing)

def dataset_roots():
    if not DATA_PARENT.is_dir():
        return []
    return sorted({folder.parent for folder in DATA_PARENT.rglob("metadata")
                   if folder.is_dir()
                   and (folder / "dish_metadata_cafe1.csv").is_file()
                   and (folder / "ingredients_metadata.csv").is_file()
                   and (folder.parent / "dish_ids").is_dir()
                   and (folder.parent / "imagery/realsense_overhead").is_dir()})


roots = dataset_roots()
if not roots:
    source = DRIVE / "nutrition5k_rgb.zip"
    assert source.is_file(), f"Falta: {source}"
    local = Path("/content/nutrition5k_rgb_v3.zip")
    shutil.copy2(source, local)
    extract_zip_safe(local, DATA_PARENT)
    roots = dataset_roots()
assert len(roots) == 1, f"Raíz RGB no única: {roots}"
DATA = roots[0]

labels_zip = V3 / "data/processed_100_grouped_provisional.zip"
assert labels_zip.is_file(), f"Falta subir: {labels_zip}"
if not (PROCESSED / "records.jsonl").is_file():
    extract_zip_safe(labels_zip, LABELS_PARENT)
package = json.loads((LABELS_PARENT / "package_manifest.json").read_text())
for name, digest in package["file_sha256"].items():
    path = PROCESSED / name
    assert path.is_file() and hashlib.sha256(path.read_bytes()).hexdigest() == digest, f"Etiquetas modificadas: {name}"
summary = json.loads((PROCESSED / "summary.json").read_text())
assert summary["mode"] == "provisional" and summary["official_test_preserved"]
print("RGB:", DATA)
print("Etiquetas:", PROCESSED)
print("Splits:", summary["samples"], "| Media train:", summary["train_mean_mass_grams"], "g")
print("Advertencia: agrupamiento provisional, no independencia certificada.")


## 5. Snapshot opcional de V2 — sin modificar el original

V3 no escribe en V2. Este respaldo adicional copia runs/reports/predictions/weights en un ZIP nuevo con hashes.
No hacerlo mientras V2 siga entrenando. El ZIP puede ocupar bastante espacio; no incluye el cache de profundidad.


In [ ]:
RUN_BACKUP_V2 = False
V2_SNAPSHOT = DRIVE / "backups/v2_before_v3.zip"
if RUN_BACKUP_V2:
    assert V2.is_dir(), f"No existe {V2}"
    assert not V2_SNAPSHOT.exists(), "Ya existe ese respaldo: elegir otro nombre."
    run_stream([sys.executable, "-u", "scripts/archive_experiment.py", "--source", V2, "--output", V2_SNAPSHOT])
else:
    print("Respaldo desactivado. V2 se conserva en su carpeta original.")


## 6. Depth Anything V2 congelado: código y pesos

Se usa el mismo estimador vits, pero otra geometría. Puede copiar sus pesos desde V2; no sus mapas 224×224.
La copia/descarga no entrena el modelo. Los pesos se guardan en v3/weights/.


In [ ]:
import urllib.request
depth_config = json.loads(CONFIG.read_text())["depth"]
if not (DEPTH_REPO / ".git").is_dir():
    subprocess.run(["git", "clone", "https://github.com/DepthAnything/Depth-Anything-V2.git", str(DEPTH_REPO)], check=True)
    subprocess.run(["git", "-C", str(DEPTH_REPO), "checkout", depth_config["repo_revision"]], check=True)
revision = subprocess.check_output(["git", "-C", str(DEPTH_REPO), "rev-parse", "HEAD"], text=True).strip()
assert revision == depth_config["repo_revision"], "El código de Depth Anything no coincide con el commit fijado."
DEPTH_WEIGHTS.parent.mkdir(parents=True, exist_ok=True)
if not DEPTH_WEIGHTS.is_file():
    previous = V2 / "weights/depth_anything_v2_vits.pth"
    if previous.is_file():
        shutil.copy2(previous, DEPTH_WEIGHTS)
    else:
        url = "https://huggingface.co/depth-anything/Depth-Anything-V2-Small/resolve/main/depth_anything_v2_vits.pth"
        urllib.request.urlretrieve(url, DEPTH_WEIGHTS)
print("Pesos depth:", DEPTH_WEIGHTS)


## 7. Cache nuevo de profundidad y controles previos

Primera sesión: RUN_DEPTH_PREPARATION=True. Se generan mapas 320×320 desde RGB completo con letterbox.
Sesiones siguientes: dejar False para restaurar v3/depth_v3_320.zip.
Este paso no entrena ConvNeXt. Nunca se cargan mapas del sensor ni el cache de V2.


In [ ]:
RUN_DEPTH_PREPARATION = False
RUN_SHAPE_CHECK = False
if not (DEPTH_CACHE / "manifest.json").is_file() and DEPTH_ARCHIVE.is_file():
    local = Path("/content/depth_v3_320.zip")
    shutil.copy2(DEPTH_ARCHIVE, local)
    extract_zip_safe(local, DEPTH_CACHE.parent)
if RUN_DEPTH_PREPARATION:
    run_stream([sys.executable, "-u", "scripts/prepare_depth.py", "--config", CONFIG, "--dataset-root", DATA,
                "--processed-dir", PROCESSED, "--depth-cache", DEPTH_CACHE,
                "--depth-anything-repo", DEPTH_REPO, "--depth-checkpoint", DEPTH_WEIGHTS, "--device", DEVICE])
    if not DEPTH_ARCHIVE.exists():
        local_zip = Path(shutil.make_archive("/content/depth_v3_320_export", "zip", root_dir=DEPTH_CACHE.parent, base_dir=DEPTH_CACHE.name))
        shutil.copy2(local_zip, DEPTH_ARCHIVE)
if (DEPTH_CACHE / "manifest.json").is_file():
    run_stream([sys.executable, "-u", "scripts/train_image_model.py", "--config", CONFIG, "--dataset-root", DATA,
                "--processed-dir", PROCESSED, "--depth-cache", DEPTH_CACHE, "--check-data", "--device", DEVICE])
else:
    print("Falta cache 320×320: habilitar RUN_DEPTH_PREPARATION antes de entrenar.")
if RUN_SHAPE_CHECK:
    run_stream([sys.executable, "-u", "scripts/inspect_image_model.py", "--config", CONFIG])


## 8. Entrenamiento V3 — desactivado

Primera corrida: MODE="new", carpeta RUN nueva. Nunca cargar/resumir el checkpoint de V2.
Máximo 100 épocas, paciencia 8 por L1 de ingredientes. Batch 2×16 = 32.
Para un chequeo corto, TOTAL_EPOCHS=5 y luego reanudar ESTA V3 desde last.pt hasta 100.
El horizonte del scheduler queda fijo en 100 para no cambiar la trayectoria al reanudar.


In [ ]:
RUN_TRAINING = False
MODE = "new"
TOTAL_EPOCHS = 100
if RUN_TRAINING:
    assert (DEPTH_CACHE / "manifest.json").is_file(), "Falta cache V3."
    assert MODE in ("new", "resume")
    args = [sys.executable, "-u", "scripts/train_image_model.py", "--config", CONFIG, "--dataset-root", DATA,
            "--processed-dir", PROCESSED, "--depth-cache", DEPTH_CACHE, "--output-dir", RUN,
            "--torch-cache", DRIVE / "torch_cache", "--device", DEVICE, "--workers", "2", "--epochs", TOTAL_EPOCHS]
    if MODE == "resume":
        assert (RUN / "last.pt").is_file(), "Falta last.pt de esta V3."
        args += ["--resume", RUN / "last.pt"]
    else:
        assert not RUN.exists() or not any(RUN.iterdir()), "RUN ya tiene resultados: usar resume de V3 o otra carpeta nueva."
    run_stream(args)
else:
    print("Entrenamiento V3 desactivado.")


## 9. Calibrar presencia con validación y evaluar test

Elegir REPORT_TAG nuevo al reevaluar: se preservan reportes anteriores.
La calibración elige un umbral de presencia, no modifica gramos ni garantiza probabilidades calibradas.
Se registran época, hash del checkpoint y fingerprints de etiquetas/perfiles para comparar.


In [ ]:
RUN_CALIBRATION = False
RUN_TEST = False
shared = ["--dataset-root", DATA, "--processed-dir", PROCESSED, "--depth-cache", DEPTH_CACHE,
          "--device", DEVICE, "--batch-size", "2", "--workers", "2"]
if RUN_CALIBRATION:
    assert (RUN / "best.pt").is_file(), "Falta best.pt de V3."
    run_stream([sys.executable, "-u", "scripts/evaluate_image_model.py", "--checkpoint", RUN / "best.pt",
                "--split", "validation", "--calibrate", "--output-dir", VALIDATION, *shared])
if RUN_TEST:
    assert CHECKPOINT.is_file(), "Primero calibrar con validación."
    run_stream([sys.executable, "-u", "scripts/evaluate_image_model.py", "--checkpoint", CHECKPOINT,
                "--split", "test", "--output-dir", TEST, *shared])
for split, folder in (("VALIDACIÓN", VALIDATION), ("TEST", TEST)):
    if (folder / "metrics.json").is_file():
        print(split); display(json.loads((folder / "metrics.json").read_text()))
if not RUN_CALIBRATION and not RUN_TEST:
    print("Calibración y evaluación desactivadas.")


## 10. Inferencia RGB y tarjetas real vs estimado

Usar "personal", "validation" o "test". Las referencias se usan SOLO al mostrar resultados, nunca para predecir.
Para fotos personales reutiliza photos/annotations.json. Los resultados están en v3/predictions/REPORT_TAG/.


In [ ]:
RUN_INFERENCE = False
PHOTO_SOURCE = "test"
PLATE_INDEX = 0
PHOTO = DRIVE / "photos/IMG_9993.png"
if PHOTO_SOURCE in ("validation", "test"):
    records = [json.loads(line) for line in (PROCESSED / "records.jsonl").read_text().splitlines() if line.strip()]
    candidates = [r for r in records if r["split"] == PHOTO_SOURCE]
    assert 0 <= PLATE_INDEX < len(candidates)
    selected = candidates[PLATE_INDEX]
    PHOTO = Path("/content/v3_examples") / f"{selected['dish_id']}.png"
    PHOTO.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(DATA / selected["image"], PHOTO)
    print("Plato:", selected["dish_id"], "| Split:", PHOTO_SOURCE)
elif PHOTO_SOURCE != "personal":
    raise ValueError("PHOTO_SOURCE debe ser personal, validation o test.")
if RUN_INFERENCE:
    assert CHECKPOINT.is_file() and PHOTO.is_file()
    run_stream([sys.executable, "-u", "scripts/infer_image_model.py", "--checkpoint", CHECKPOINT, "--image", PHOTO,
                "--output-dir", PREDICTIONS, "--device", DEVICE, "--save-attention", "--top-k", "10",
                "--depth-anything-repo", DEPTH_REPO, "--depth-checkpoint", DEPTH_WEIGHTS])
else:
    print("Inferencia V3 desactivada.")


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import pandas as pd

from IPython.display import HTML, display
from html import escape
import math

# Referencias solo para comparar: nunca se pasan al modelo.
def load_reference(photo, source):
    if source in ("validation", "test"):
        records_path = PROCESSED / "records.jsonl"
        if not records_path.is_file():
            return None, "No se encontró records.jsonl: solo se muestran estimaciones."
        matches = [r for r in (json.loads(line) for line in records_path.read_text().splitlines() if line.strip())
                   if r["dish_id"] == photo.stem and r["split"] == source]
        if len(matches) != 1:
            return None, "La foto no coincide con un plato único del split elegido."
        record = matches[0]
        return {"total_mass_grams": record["total_mass"], "nutrition": record["nutrients"],
                "ingredients": record["ingredients"], "dataset": True}, "Referencia: anotaciones de Nutrition5k."
    for path in (DRIVE / "photos/annotations.json", Path("photos/annotations.json")):
        if path.is_file():
            matches = [r for r in json.loads(path.read_text()).get("photos", [])
                       if Path(r["image"]).stem.casefold() == photo.stem.casefold()]
            if len(matches) == 1:
                return matches[0], "Referencia: pesos informados por el usuario; nutrientes solo si están anotados."
    return None, "Sin referencia para esta foto: no se calculan errores."

def valid_number(value):
    return value is not None and math.isfinite(float(value))

def fmt(value, unit):
    return f"{float(value):,.1f} {unit}".replace(",", "X").replace(".", ",").replace("X", ".") if valid_number(value) else "Sin dato"

def comparison_card(label, unit, estimate, actual):
    known = valid_number(actual) and valid_number(estimate)
    delta = float(estimate) - float(actual) if known else None
    direction = "Sobreestimación" if known and delta > 0 else "Subestimación" if known and delta < 0 else "Sin diferencia" if known else "Sin referencia"
    signed = f"{delta:+.1f} {unit}".replace(".", ",") if known else "—"
    percent = f"{100 * abs(delta) / abs(float(actual)):.1f} %".replace(".", ",") if known and float(actual) != 0 else "No definido (real = 0)" if known else "—"
    return f"""
    <article class="ime-card">
      <h4>{escape(label)}</h4>
      <div class="ime-values">
        <div class="ime-real"><span>REAL</span><strong>{fmt(actual, unit)}</strong></div>
        <div class="ime-est"><span>ESTIMADO</span><strong>{fmt(estimate, unit)}</strong></div>
      </div>
      <div class="ime-delta">{direction} · Δ {signed}</div>
      <div class="ime-detail">Error absoluto: {fmt(abs(delta) if known else None, unit)} · {percent}</div>
    </article>"""

def render_comparison(prediction, reference, note):
    real = reference or {}
    nutrition = real.get("nutrition") or real.get("nutrients") or {}
    cards = comparison_card("Masa total", "g", prediction["predicted_total_mass_grams"], real.get("total_mass_grams"))
    for key, label, unit in (("calories", "Calorías", "kcal"), ("protein", "Proteínas", "g"),
                             ("carbohydrates", "Carbohidratos", "g"), ("fat", "Grasas", "g")):
        cards += comparison_card(label, unit, prediction["nutrition"].get(key), nutrition.get(key))
    styles = """
    <style>
    .ime-comparison {font-family:Arial,sans-serif;color:#172b45;margin:16px 0;}
    .ime-comparison h3 {margin:0 0 8px;font-size:22px;}
    .ime-comparison p {line-height:1.5;}
    .ime-grid {display:grid;grid-template-columns:repeat(auto-fit,minmax(min(100%,260px),1fr));gap:14px;}
    .ime-card {background:#f5f8fc;border:1px solid #dce5ef;border-radius:24px;padding:18px;min-width:0;}
    .ime-card h4 {margin:0 0 14px;font-size:17px;overflow-wrap:anywhere;}
    .ime-values {display:flex;flex-wrap:wrap;gap:8px;}
    .ime-values>div {flex:1;min-width:90px;border-radius:18px;padding:12px;}
    .ime-real {background:#e0f2ee;color:#14584b;}
    .ime-est {background:#e5edff;color:#234e94;}
    .ime-values span {display:block;font-size:11px;letter-spacing:1px;margin-bottom:6px;}
    .ime-values strong {display:block;font-size:22px;font-variant-numeric:tabular-nums;overflow-wrap:anywhere;}
    .ime-delta {display:inline-block;margin-top:12px;padding:7px 11px;border-radius:18px;background:#fff;color:#172b45;font-size:13px;}
    .ime-detail {margin-top:10px;font-size:13px;color:#40546c;line-height:1.5;}
    .ime-reference {display:flex;flex-wrap:wrap;gap:8px;}
    .ime-reference span {padding:10px 14px;background:#e0f2ee;border-radius:20px;color:#14584b;}
    </style>"""
    title = f"Real vs. estimado · {escape(PHOTO.name)}"
    body = f"<h3>{title}</h3><p>{escape(note)}<br>Δ = estimado − real. El porcentaje usa el valor real como denominador.</p><div class='ime-grid'>{cards}</div>"
    if prediction.get("mass_source") == "user":
        body += "<p>Atención: los ingredientes y nutrientes usaron una masa ingresada manualmente; la tarjeta de masa muestra la predicción del modelo.</p>"
    all_ingredients = prediction["ingredients"]
    if real.get("dataset"):
        by_key = {str(item["key"]): item for item in all_ingredients}
        real_grams = {key: 0.0 for key in by_key}
        for component in real["ingredients"]:
            key = str(component["ingredient_id"])
            mapped = key if key in by_key else "__other__"
            if mapped not in real_grams:
                raise ValueError("Falta la clase other en prediction.json para comparar ingredientes.")
            real_grams[mapped] += float(component["grams"])
        order = sorted(by_key, key=lambda key: max(real_grams[key], float(by_key[key]["grams"])), reverse=True)[:10]
        ingredient_cards = "".join(comparison_card(by_key[key]["name"], "g", by_key[key]["grams"], real_grams[key]) for key in order)
        l1 = sum(abs(float(by_key[key]["grams"]) - real_grams[key]) for key in by_key)
        body += f"<h3 style='margin-top:24px'>Masas por ingrediente</h3><p>Error L1 del plato (todas las clases): {fmt(l1, 'g')}. Se muestran los 10 ingredientes con mayor masa real o estimada; los que quedan fuera del vocabulario se agrupan en other.</p><div class='ime-grid'>{ingredient_cards}</div>"
    elif real.get("ingredients"):
        chips = "".join(f"<span>{escape(item['name'])}: <strong>{fmt(item['grams'], 'g')}</strong></span>" for item in real["ingredients"])
        body += f"<h3 style='margin-top:24px'>Ingredientes reales informados</h3><div class='ime-reference'>{chips}</div><p>No se comparan automáticamente por nombre con las clases del modelo: se necesita una correspondencia validada.</p>"
    display(HTML(styles + f"<section class='ime-comparison'>{body}</section>"))

result_dir = PREDICTIONS / PHOTO.stem
prediction_file = result_dir / "prediction.json"
if prediction_file.is_file():
    prediction = json.loads(prediction_file.read_text())
    reference, reference_note = load_reference(PHOTO, PHOTO_SOURCE)
    render_comparison(prediction, reference, reference_note)
    ingredients = sorted(prediction["ingredients"], key=lambda item: item["grams"], reverse=True)[:10]
    fig, axes = plt.subplots(1, 3, figsize=(17, 5))
    axes[0].imshow(Image.open(result_dir / "intermediates/input_crop.png"))
    axes[0].set_title("RGB usado por el modelo"); axes[0].axis("off")
    axes[1].imshow(Image.open(result_dir / "intermediates/estimated_relative_depth.png"), cmap="magma")
    axes[1].set_title("Profundidad relativa estimada"); axes[1].axis("off")
    axes[2].barh([item["name"] for item in ingredients][::-1],
                [item["grams"] for item in ingredients][::-1], color="#278b87")
    axes[2].set_xlabel("Gramos estimados"); axes[2].set_title("Top 10 ingredientes")
    plt.suptitle(f"Masa total estimada: {prediction['predicted_total_mass_grams']:.1f} g", fontsize=16)
    plt.tight_layout(); plt.show()
    display(pd.DataFrame(ingredients)[["name", "probability", "recognized", "grams", "share"]])
    overlays = sorted((result_dir / "intermediates").glob("*_attention.png"))[:10]
    if overlays:
        fig, axes = plt.subplots(2, 5, figsize=(18, 7))
        for ax, path in zip(axes.flat, overlays):
            ax.imshow(Image.open(path)); ax.set_title(path.stem.replace("_attention", "")); ax.axis("off")
        for ax in list(axes.flat)[len(overlays):]:
            ax.axis("off")
        plt.suptitle("Atención combinada de las etapas 3 y 4")
        plt.tight_layout(); plt.show()
else:
    print("Primero ejecutá la inferencia del punto 10.")


## 11. Diagnóstico V3 — solo resultados guardados

Métricas, curvas, pérdidas separadas, clases ignoradas, errores condicionales, gramos falsos, platos difíciles y atención.
No ejecuta el modelo. La geometría V3 usa 500 tokens: etapa 3 con 400 y etapa 4 con 100.
Los resultados no prueban causalidad ni independencia completa del split provisional.


In [ ]:
# Solo lectura: no carga pesos, no ejecuta el modelo y no escribe archivos.
from pathlib import Path
from datetime import datetime
from collections import Counter
from html import escape
import csv, json
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML, display

TOP_DIAGNOSTIC = 12
MIN_CLASS_SUPPORT = 5
SHOW_TEST_SUMMARY = True  # Solo referencia global; diagnósticos y umbrales usan VALIDACIÓN.

def diag_json(path):
    try:
        return json.loads(Path(path).read_text(encoding="utf-8")) if Path(path).is_file() else None
    except (OSError, ValueError) as exc:
        diag_note(f"No se pudo leer {path}: {exc}")
        return None

def diag_note(message):
    display(HTML(f"<p class='ime11-note'>{escape(str(message))}</p>"))

def diag_title(title):
    display(HTML(f"<h3 class='ime11-heading'>{escape(title)}</h3>"))

def diag_fmt(value, unit="", digits=2):
    if value is None or isinstance(value, (float, np.floating)) and not np.isfinite(value):
        return "—"
    return f"{float(value):.{digits}f} {unit}".strip().replace(".", ",")

def diag_table(rows, limit=None):
    rows = list(rows)
    if not rows:
        diag_note("No hay datos para esta tabla.")
        return
    headers = list(rows[0])
    shown = rows[:limit] if limit is not None else rows
    body = "".join("<tr>" + "".join(f"<td>{escape(str(row.get(key, '—')))}</td>" for key in headers) + "</tr>" for row in shown)
    display(HTML("<div class='ime11-scroll'><table class='ime11-table'><thead><tr>" +
                 "".join(f"<th>{escape(key)}</th>" for key in headers) + "</tr></thead><tbody>" + body + "</tbody></table></div>"))

def diag_cards(items):
    cards = "".join(f"<article class='ime11-card'><span>{escape(label)}</span><strong>{escape(value)}</strong><small>{escape(note)}</small></article>"
                    for label, value, note in items)
    display(HTML(f"<div class='ime11-grid'>{cards}</div>"))

def diag_prediction_arrays(path):
    required = ("probabilities", "predicted_shares", "predicted_total_mass", "presence",
                "masses", "total_mass", "nutrients", "dish_ids", "vocabulary_keys", "conditional_grams")
    with np.load(path, allow_pickle=False) as source:
        missing = set(required) - set(source.files)
        if missing:
            raise ValueError(f"Faltan arrays en predictions.npz: {sorted(missing)}")
        arrays = {key: source[key] for key in required}
    n, classes = arrays["masses"].shape
    if n == 0 or classes == 0:
        raise ValueError("predictions.npz está vacío.")
    for key in ("probabilities", "predicted_shares", "presence", "conditional_grams"):
        if arrays[key].shape != (n, classes):
            raise ValueError(f"Dimensiones incompatibles en {key}.")
    for key in ("predicted_total_mass", "total_mass", "dish_ids"):
        if arrays[key].shape != (n,):
            raise ValueError(f"Dimensiones incompatibles en {key}.")
    if arrays["nutrients"].shape != (n, 4) or arrays["vocabulary_keys"].shape != (classes,):
        raise ValueError("Dimensiones incompatibles de nutrientes o vocabulario.")
    if len(set(map(str, arrays["vocabulary_keys"]))) != classes or len(set(map(str, arrays["dish_ids"]))) != n:
        raise ValueError("IDs de clases o platos duplicados en la evaluación.")
    for key in required:
        if key not in ("dish_ids", "vocabulary_keys") and not np.isfinite(arrays[key]).all():
            raise ValueError(f"Valores no finitos en {key}.")
    if np.any(arrays["masses"] < 0) or np.any(arrays["total_mass"] <= 0) or np.any(arrays["predicted_total_mass"] <= 0):
        raise ValueError("Masas inválidas.")
    for key in ("probabilities", "presence", "predicted_shares"):
        if np.any(arrays[key] < 0) or np.any(arrays[key] > 1):
            raise ValueError(f"{key} debe estar en [0, 1].")
    if not np.allclose(arrays["predicted_shares"].sum(axis=1), 1, atol=1e-3):
        raise ValueError("Las proporciones predichas no suman 1.")
    return arrays

def diag_statistics(arrays, threshold):
    truth = arrays["masses"] > 0
    if not np.array_equal(truth, arrays["presence"] > 0.5):
        raise ValueError("Las etiquetas de presencia y gramos no coinciden.")
    predicted = arrays["predicted_shares"] * arrays["predicted_total_mass"][:, None]
    absolute = np.abs(predicted - arrays["masses"])
    false_mass = predicted * ~truth
    oracle = arrays["predicted_shares"] * arrays["total_mass"][:, None]
    result = dict(truth=truth, predicted=predicted, absolute=absolute, false_mass=false_mass,
                  l1=absolute.sum(axis=1), oracle_l1=np.abs(oracle - arrays["masses"]).sum(axis=1),
                  mass_error=arrays["predicted_total_mass"] - arrays["total_mass"])
    if threshold is not None:
        if not 0 <= float(threshold) <= 1:
            raise ValueError("Umbral fuera de [0,1].")
        detected = arrays["probabilities"] >= threshold
        tp = (detected & truth).sum(axis=0)
        fp = (detected & ~truth).sum(axis=0)
        fn = (~detected & truth).sum(axis=0)
        precision = np.divide(tp, tp + fp, out=np.zeros_like(tp, dtype=float), where=(tp + fp) > 0)
        recall = np.divide(tp, tp + fn, out=np.zeros_like(tp, dtype=float), where=(tp + fn) > 0)
        f1 = np.divide(2 * precision * recall, precision + recall,
                       out=np.zeros_like(precision), where=(precision + recall) > 0)
        result.update(detected=detected, tp=tp, fp=fp, fn=fn, precision=precision, recall=recall, f1=f1,
                      exact=np.all(detected == truth, axis=1))
    return result

def diag_class_rows(arrays, stats, vocabulary, records):
    keys = list(map(str, arrays["vocabulary_keys"]))
    labels = {str(entry["key"]): entry["name"] for entry in (vocabulary or {}).get("entries", [])}
    train_counts = Counter()
    if records:
        for record in records:
            if record["split"] == "train":
                train_counts.update({str(c["ingredient_id"]) if str(c["ingredient_id"]) in keys else "__other__"
                                     for c in record["ingredients"] if float(c["grams"]) > 0})
    rows = []
    for index, key in enumerate(keys):
        present = stats["truth"][:, index]
        absent = ~present
        row = {"Clase": labels.get(key, key), "Clave": key,
               "Train platos": train_counts[key] if records else "—", "Val platos": int(present.sum()),
               "MAE condicional presente g": float(np.abs(arrays["conditional_grams"][present, index] - arrays["masses"][present, index]).mean()) if present.any() else None,
               "Gate medio presente %": float(100 * arrays["probabilities"][present, index].mean()) if present.any() else None,
               "L1 aportado g/plato": float(stats["absolute"][:, index].mean()),
               "Gramos falsos g/plato": float(stats["false_mass"][:, index].mean()),
               "MAE presente g": float(stats["absolute"][present, index].mean()) if present.any() else None,
               "Gramos falsos por ausencia": float(stats["predicted"][absent, index].mean()) if absent.any() else None}
        if "f1" in stats:
            row.update({"Precisión %": float(100 * stats["precision"][index]),
                        "Recall %": float(100 * stats["recall"][index]), "F1 %": float(100 * stats["f1"][index]),
                        "FP": int(stats["fp"][index]), "FN": int(stats["fn"][index])})
        rows.append(row)
    return rows

def diag_display_classes(rows):
    return [{key: diag_fmt(value) if isinstance(value, (float, np.floating)) or value is None else value
             for key, value in row.items()} for row in rows]

def diag_nutrition_rows(arrays, stats, vocabulary):
    if not vocabulary or list(map(str, arrays["vocabulary_keys"])) != [str(e["key"]) for e in vocabulary["entries"]]:
        raise ValueError("No se puede asociar la matriz nutricional: el vocabulario no coincide en orden y claves.")
    order = ["calories", "fat", "carbohydrates", "protein"]
    if vocabulary["nutrient_order"] != order:
        raise ValueError("Orden de nutrientes incompatible.")
    matrix = np.array([[e["nutrients_per_gram"][name] for name in order] for e in vocabulary["entries"]])
    cases = {"End-to-end": stats["predicted"],
             "Masa total real": arrays["predicted_shares"] * arrays["total_mass"][:, None],
             "Gramos reales + perfiles del vocabulario": arrays["masses"]}
    rows = []
    for case, masses in cases.items():
        predicted = masses @ matrix
        for index, name in enumerate(order):
            error = predicted[:, index] - arrays["nutrients"][:, index]
            denominator = np.abs(arrays["nutrients"][:, index]).sum()
            rows.append({"Escenario": case, "Nutriente": name, "Unidad": "kcal" if name == "calories" else "g",
                         "MAE": float(np.abs(error).mean()), "Sesgo estimado − real": float(error.mean()),
                         "PMAE %": float(100 * np.abs(error).sum() / denominator) if denominator > 0 else None})
    return rows

def diag_history(path, config):
    if not Path(path).is_file():
        diag_note("Falta history.csv: no se puede diagnosticar evolución ni early stopping.")
        return
    with Path(path).open(encoding="utf-8-sig", newline="") as incoming:
        rows = [{key: float(value) for key, value in row.items()
                 if key != "encoder_trainable" and value not in ("", None)} for row in csv.DictReader(incoming)]
    if not rows:
        diag_note("history.csv está vacío.")
        return
    rows.sort(key=lambda r: r["epoch"])
    if len({r["epoch"] for r in rows}) != len(rows):
        diag_note("Hay épocas duplicadas: revisar historial antes de interpretar tendencias.")
        return
    selection = config.get("selection_metric", "ingredient_mass_l1_per_plate")
    if selection not in rows[0]:
        diag_note(f"La métrica de selección {selection} no está en history.csv.")
        return
    best = (max if selection == "micro_f1" else min)(rows, key=lambda r: r[selection])
    stale = int(rows[-1]["epoch"] - best["epoch"])
    diag_cards([("Última época", str(int(rows[-1]["epoch"])), "history.csv"),
                ("Mejor época registrada", str(int(best["epoch"])), f"Seleccionada por {selection}; no certifica la época del reporte calibrado"),
                ("Épocas sin nuevo mejor", str(stale), f"Patience configurada: {config.get('patience', '—')}; aplica después de warmup")])
    blocks = []
    for start in range(max(0, len(rows) - 30), len(rows), 10):
        block = rows[start:start + 10]
        blocks.append({"Épocas": f"{int(block[0]['epoch'])}–{int(block[-1]['epoch'])}",
                      **{key: diag_fmt(np.mean([r[key] for r in block])) for key in
                         ("train_loss", "validation_loss", "ingredient_mass_l1_per_plate", "total_mass_mae", "calorie_mae")},
                      "Brecha val − train": diag_fmt(np.mean([r["validation_loss"] - r["train_loss"] for r in block]))})
    diag_table(blocks)
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    epochs = [r["epoch"] for r in rows]
    for key, label in (("train_loss", "Train"), ("validation_loss", "Validación")):
        axes[0].plot(epochs, [r[key] for r in rows], label=label)
    axes[0].set_title("Pérdida total"); axes[0].legend()
    for key, label in (("ingredient_mass_l1_per_plate", "L1 ingredientes"), ("total_mass_mae", "MAE masa")):
        axes[1].plot(epochs, [r[key] for r in rows], label=label)
    axes[1].set_title("Validación · gramos"); axes[1].legend()
    axes[2].plot(epochs, [100 * r["micro_f1"] for r in rows])
    axes[2].set_title("F1 micro · umbral durante entrenamiento (%)")
    for ax in axes:
        ax.axvline(best["epoch"], color="gray", linestyle=":", label="Mejor época")
        ax.set_xlabel("Época"); ax.grid(alpha=0.2)
    fig.tight_layout(); plt.show(); plt.close(fig)
    diag_note("La F1 del historial usa el umbral de entrenamiento; no compararla directamente con F1 calibrada. "
              "Train incluye augmentations/dropout y validación no: la brecha de pérdidas es orientativa, no una prueba aislada de sobreajuste.")

def diag_inference_tensors(folder):
    diag_title("11.7 Atención y profundidad del último ejemplo guardado")
    path = Path(folder) / "intermediates/tensors.npz"
    if not path.is_file():
        diag_note("Sin tensores del punto 10. Este bloque es opcional; no se ejecuta inferencia para generarlos.")
        return
    with np.load(path, allow_pickle=False) as source:
        summary = diag_json(Path(folder) / "intermediates/tensor_summary.json") or {}
        wanted = ("depth_input", "rgb_features_stage3", "rgb_features_stage4", "depth_features_stage3",
                  "depth_features_stage4", "fused_stage3", "fused_stage4", "spatial_memory")
        diag_table([{"Tensor": key, "Shape": str(summary[key]["shape"]), "Mínimo": diag_fmt(summary[key]["min"]),
                     "Máximo": diag_fmt(summary[key]["max"]), "Media": diag_fmt(summary[key]["mean"])}
                    for key in wanted if key in summary])
        if "attention_tokens" not in source.files:
            diag_note("No hay attention_tokens para comparar etapas.")
            return
        attention = np.asarray(source["attention_tokens"][0], dtype=float)
        if attention.ndim != 2 or attention.shape[1] != 500 or not np.isfinite(attention).all() or np.any(attention < 0):
            diag_note("Forma/valores de atención inesperados; no se inventan estadísticas.")
            return
        totals = attention.sum(axis=1, keepdims=True)
        if np.any(totals <= 0):
            diag_note("Hay consultas con atención nula: no se calculan porcentajes/entropía.")
            return
        attention = attention / totals
        entropy = -(attention * np.log(np.maximum(attention, 1e-12))).sum(axis=1) / np.log(attention.shape[1])
        names = source["ingredient_names"]
        masses = source["ingredient_masses"][0]
        probabilities = source["presence_probabilities"][0]
        diag_table([{"Consulta": str(names[i]), "Gramos estimados": diag_fmt(masses[i]),
                     "Probabilidad %": diag_fmt(100 * probabilities[i]),
                     "Atención etapa 3 %": diag_fmt(100 * attention[i, :400].sum()),
                     "Atención etapa 4 %": diag_fmt(100 * attention[i, 400:].sum()),
                     "Entropía normalizada": diag_fmt(entropy[i], digits=3)}
                    for i in np.argsort(masses)[::-1][:TOP_DIAGNOSTIC]])
    diag_note("Un único ejemplo no evalúa el módulo. Etapa 3 tiene 400 tokens y etapa 4 tiene 100: "
              "atención uniforme reparte 80%/20%. Entropía alta = atención difusa; baja = concentrada, no necesariamente correcta. "
              "La profundidad es relativa, no centímetros ni volumen. La utilidad de esta rama necesita ablaciones entrenadas.")

def run_v3_diagnostics():
    display(HTML("""<style>
    .ime11-heading {font-family:Arial,sans-serif;color:#172b45;margin-top:26px;}
    .ime11-note {font-family:Arial,sans-serif;line-height:1.6;color:#40546c;}
    .ime11-grid {display:grid;grid-template-columns:repeat(auto-fit,minmax(min(100%,220px),1fr));gap:12px;margin:14px 0;}
    .ime11-card {font-family:Arial,sans-serif;background:#edf4fc;color:#172b45;border:1px solid #dce5ef;border-radius:24px;padding:18px;min-width:0;}
    .ime11-card span,.ime11-card small {display:block;line-height:1.5;overflow-wrap:anywhere;}
    .ime11-card strong {display:block;font-size:26px;margin:8px 0;font-variant-numeric:tabular-nums;}
    .ime11-scroll {overflow-x:auto;margin:12px 0;}
    .ime11-table {font-family:Arial,sans-serif;border-collapse:collapse;width:100%;color:#172b45;background:#fff;}
    .ime11-table th {background:#e5edff;text-align:left;}
    .ime11-table th,.ime11-table td {padding:10px;border-bottom:1px solid #dce5ef;white-space:nowrap;}
    </style>"""))
    diag_title("11.1 Fuentes, configuración y vigencia")
    run_config = diag_json(RUN / "config.json")
    config = run_config or diag_json(CONFIG) or {}
    if not run_config:
        diag_note("No se encontró config.json de la corrida: la configuración mostrada es la plantilla, no certifica el entrenamiento.")
    paths = {"Config corrida": RUN / "config.json", "Historial": RUN / "history.csv",
             "Métricas validación": VALIDATION / "metrics.json", "Detalles validación": VALIDATION / "evaluation_details.json",
             "Predicciones validación": VALIDATION / "predictions.npz", "Vocabulario": PROCESSED / "vocabulary.json",
             "Registros": PROCESSED / "records.jsonl", "Última validación": RUN / "latest_validation.json"}
    diag_table([{"Archivo": label, "Ruta": str(path), "Estado": "Disponible" if path.is_file() else "Falta",
                 "Modificado (hora runtime)": datetime.fromtimestamp(path.stat().st_mtime).strftime("%Y-%m-%d %H:%M") if path.is_file() else "—"}
                for label, path in paths.items()])
    if (RUN / "best.pt").is_file() and (VALIDATION / "metrics.json").is_file() and (RUN / "best.pt").stat().st_mtime > (VALIDATION / "metrics.json").stat().st_mtime:
        diag_note("ALERTA: best.pt es más reciente que el reporte de validación. Podría faltar recalibrar/evaluar el nuevo mejor; no se hace automáticamente.")
    diag_note("Los reportes actuales no registran época ni hash del checkpoint: fechas iguales/cercanas no garantizan identidad. "
              "No mezclar latest_validation (última época, umbral de entrenamiento) con el reporte calibrado. Test solo se muestra como referencia.")
    depth = config.get("depth", {})
    diag_table([{"Componente": "Encoder RGB", "Configuración": f"{config.get('encoder', '—')} · {config.get('encoder_training', '—')}; V3 congela stem/etapa 1"},
                {"Componente": "Fusión y atención", "Configuración": f"RGB etapas 3+4 · residual gated depth · 500 tokens · {config.get('top_k', '—')} consultas · dim {config.get('attention_dim', '—')} · {config.get('attention_layers', '—')} capas · {config.get('attention_heads', '—')} heads"},
                {"Componente": "Depth Anything", "Configuración": f"{depth.get('encoder', '—')} · congelado={depth.get('frozen', '—')} · normalización {depth.get('normalization', '—')}; CNN de profundidad/fusión entrenables"},
                {"Componente": "MLP", "Configuración": f"768 → {config.get('shared_dims', '—')} · dropout {config.get('dropout', '—')}"},
                {"Componente": "Optimización", "Configuración": f"LR heads {config.get('learning_rate', '—')} · LR encoder {config.get('encoder_learning_rate', '—')} · batch efectivo {config.get('batch_size', 0) * config.get('accumulation_steps', 1)} · seed {config.get('seed', '—')}"},
                {"Componente": "Pérdidas", "Configuración": f"{config.get('presence_loss', '—')} · masa / {config.get('mass_scale', 'media train no guardada')} g · ingredientes / {config.get('ingredient_mass_scale', '—')} g · pesos {config.get('loss_weights', {})}"}])
    diag_title("11.2 Evolución y checkpoint seleccionado")
    diag_history(RUN / "history.csv", config)
    latest = diag_json(RUN / "latest_validation.json")
    if latest:
        weights = config.get("loss_weights", {})
        parts = latest.get("losses", {})
        diag_table([{"Pérdida (última validación)": key, "Valor sin peso": diag_fmt(value, digits=4),
                     "Peso": weights.get(key, "—"), "Contribución ponderada": diag_fmt(value * weights[key], digits=4) if key in weights else "Total"}
                    for key, value in parts.items()])
        diag_note(f"Desglose de la época {latest.get('epoch', '—')}. El tamaño de una pérdida no mide el tamaño de sus gradientes; "
                  "no permite decidir por sí solo qué peso aumentar. El historial no guarda cada componente por época.")
    diag_title("11.3 Métricas globales y masa real como diagnóstico")
    validation = diag_json(VALIDATION / "metrics.json")
    details = diag_json(VALIDATION / "evaluation_details.json") or {}
    summaries = []
    for split, report in (("VALIDACIÓN", validation), ("TEST · referencia", diag_json(TEST / "metrics.json") if SHOW_TEST_SUMMARY else None)):
        if report:
            rec = report["ingredient_recognition"]; end = report["end_to_end_lightweight_mass"]
            summaries.append({"Split": split, "F1 micro %": diag_fmt(100 * rec["micro_f1"]),
                              "F1 macro %": diag_fmt(100 * rec["macro_f1"]), "Precision %": diag_fmt(100 * rec["micro_precision"]),
                              "Recall %": diag_fmt(100 * rec["micro_recall"]), "Exact match %": diag_fmt(100 * rec["exact_match"]),
                              "L1 ingredientes g/plato": diag_fmt(end["ingredient_mass_l1_per_plate_grams"]),
                              "MAE masa g": diag_fmt(end["total_mass_mae_grams"]), "Masa PMAE %": diag_fmt(100 * end["total_mass_pmae"]),
                              "Gramos falsos/plato": diag_fmt(end["false_ingredient_mass_per_plate_grams"])})
    diag_table(summaries)
    if validation:
        end = validation["end_to_end_lightweight_mass"]; oracle = validation["composition_with_true_total_mass"]
        l1 = end["ingredient_mass_l1_per_plate_grams"]; oracle_l1 = oracle["ingredient_mass_l1_per_plate_grams"]
        diag_cards([("Error ingredientes", diag_fmt(l1, "g/plato"), "Todas las clases: presentes y ausentes"),
                    ("Masa total real", diag_fmt(oracle_l1, "g/plato"), "Mismas proporciones, reemplazando solo masa total"),
                    ("Cambio al corregir masa", diag_fmt(l1 - oracle_l1, "g/plato"), "Positivo = mejora; no identifica por sí solo un módulo"),
                    ("Parte L1 de clases ausentes", diag_fmt(100 * end["false_ingredient_mass_per_plate_grams"] / l1, "%") if l1 > 0 else "—",
                     "Gramos asignados a ausentes / L1 total")])
        diag_note("Si poner la masa exacta casi no baja el L1, investigar reconocimiento/reparto antes de priorizar solo masa total. "
                  "Si empeora, puede existir compensación entre subestimar masa y repartir mal. Esto no prueba un fallo de atención.")
        diag_table([{"Nutriente": key, "MAE end-to-end": diag_fmt(values["mae"]), "PMAE end-to-end %": diag_fmt(100 * values["pmae"]),
                     "MAE con masa real": diag_fmt(oracle["nutrients"][key]["mae"]), "Unidad": "kcal" if key == "calories" else "g"}
                    for key, values in end["nutrients"].items()])
    diag_note("PMAE = suma de errores absolutos / suma de valores reales. No es el promedio de porcentajes individuales. "
              "El L1 de ingredientes no es el MAE de masa total: transferir gramos entre clases puede contarse dos veces.")
    records_path = PROCESSED / "records.jsonl"
    records = [json.loads(line) for line in records_path.read_text(encoding="utf-8").splitlines() if line.strip()] if records_path.is_file() else []
    vocabulary = diag_json(PROCESSED / "vocabulary.json")
    diag_title("11.4 Datos, soporte y controles del split")
    if records:
        counts = Counter(r["split"] for r in records)
        diag_table([{"Split": key, "Platos": value} for key, value in counts.items()])
        ids = [r["dish_id"] for r in records]
        diag_note(f"IDs exactos duplicados entre registros: {len(ids) - len(set(ids))}. "
                  "Esto NO audita escaneos relacionados: falta verificar su agrupación oficial entre train y validación.")
        train_mass = [float(r["total_mass"]) for r in records if r["split"] == "train"]
        if train_mass:
            diag_note(f"Media de masa train recalculada: {diag_fmt(np.mean(train_mass), 'g')}. "
                      f"Escala guardada: {diag_fmt(config.get('mass_scale'), 'g')}.")
    else:
        diag_note("Sin records.jsonl: no se puede verificar soporte de train, IDs ni escala de masa.")
    diag_title("11.5 Errores por ingrediente, coocurrencias y platos")
    prediction_path = VALIDATION / "predictions.npz"
    if prediction_path.is_file():
        try:
            arrays = diag_prediction_arrays(prediction_path)
            if records:
                expected_ids = {r["dish_id"] for r in records if r["split"] == "validation"}
                if set(map(str, arrays["dish_ids"])) != expected_ids:
                    raise ValueError("Los IDs del NPZ no coinciden con el split validation actual: no mezclar artefactos.")
            if details.get("split", "validation") != "validation":
                raise ValueError("evaluation_details.json no corresponde a validación.")
            if details.get("samples", len(arrays["dish_ids"])) != len(arrays["dish_ids"]):
                raise ValueError("El número de muestras del NPZ y evaluation_details no coincide.")
            threshold = details.get("threshold")
            diag_note(f"Umbral del reporte: {diag_fmt(threshold, digits=3)}. "
                      "Si falta evaluation_details.json, se omiten diagnósticos binarios: no se supone 0,5.")
            stats = diag_statistics(arrays, threshold)
            if validation:
                expected_l1 = validation["end_to_end_lightweight_mass"]["ingredient_mass_l1_per_plate_grams"]
                if not np.isclose(stats["l1"].mean(), expected_l1, rtol=1e-5, atol=1e-4):
                    raise ValueError("El L1 del NPZ no coincide con metrics.json: posiblemente son evaluaciones diferentes.")
                if "f1" in stats:
                    tp = stats["tp"].sum(); fp = stats["fp"].sum(); fn = stats["fn"].sum()
                    recomputed_f1 = 2 * tp / max(2 * tp + fp + fn, 1)
                    if not np.isclose(recomputed_f1, validation["ingredient_recognition"]["micro_f1"], atol=1e-6):
                        raise ValueError("La F1 del NPZ/umbral no coincide con metrics.json.")
            classes = diag_class_rows(arrays, stats, vocabulary, records)
            diag_note("Ranking por contribución al L1 (g por plato de TODO el split). La suma de clases da el L1 global; "
                      "MAE presente se calcula solo cuando la clase está presente. Soporte pequeño = conclusiones inciertas.")
            diag_table(diag_display_classes(sorted(classes, key=lambda r: r["L1 aportado g/plato"], reverse=True)), TOP_DIAGNOSTIC)
            diag_note("Ranking de gramos asignados a clases ausentes; independiente del umbral de reconocimiento.")
            diag_table(diag_display_classes(sorted(classes, key=lambda r: r["Gramos falsos g/plato"], reverse=True)), TOP_DIAGNOSTIC)
            if "f1" in stats:
                default_stats = diag_statistics(arrays, 0.5)
                recognition_rows = []
                for label, result in (("Umbral fijo 0,5", default_stats), ("Umbral del reporte", stats)):
                    tp = result["tp"].sum(); fp = result["fp"].sum(); fn = result["fn"].sum()
                    recognition_rows.append({"Escenario": label, "Precision %": diag_fmt(100 * tp / max(tp + fp, 1)),
                                             "Recall %": diag_fmt(100 * tp / max(tp + fn, 1)),
                                             "F1 micro %": diag_fmt(100 * 2 * tp / max(2 * tp + fp + fn, 1))})
                diag_table(recognition_rows)
                diag_note("La calibración cambia las decisiones de presencia, no los gramos: el modelo usa gates continuos. "
                          "Una mejora de F1 por umbral no demuestra una mejora de cantidades.")
                supported = [r for r in classes if r["Val platos"] >= MIN_CLASS_SUPPORT]
                diag_note(f"Peor F1 entre clases con al menos {MIN_CLASS_SUPPORT} platos de validación (no garantiza robustez estadística).")
                diag_table(diag_display_classes(sorted(supported, key=lambda r: r["F1 %"])), TOP_DIAGNOSTIC)
                cooccurrence = (~stats["detected"] & stats["truth"]).astype(int).T @ (stats["detected"] & ~stats["truth"]).astype(int)
                pairs = [(i, j) for i, j in zip(*np.nonzero(cooccurrence))]
                pairs.sort(key=lambda pair: cooccurrence[pair], reverse=True)
                diag_table([{"Verdadero no detectado": classes[i]["Clase"], "Falso detectado en el mismo plato": classes[j]["Clase"],
                             "Platos": int(cooccurrence[i, j])} for i, j in pairs], TOP_DIAGNOSTIC)
                diag_note("Estas son coocurrencias FN→FP en un problema multilabel, no sustituciones uno-a-uno ni una matriz de confusión exclusiva.")
                exact = stats["exact"]
                diag_table([{"Grupo": label, "Platos": int(mask.sum()), "L1 medio g/plato": diag_fmt(stats["l1"][mask].mean()) if mask.any() else "—",
                             "L1 con masa real": diag_fmt(stats["oracle_l1"][mask].mean()) if mask.any() else "—"}
                            for label, mask in (("Presencia exactamente correcta", exact), ("Alguna presencia incorrecta", ~exact))])
                if records and supported:
                    fig, ax = plt.subplots(figsize=(8, 4))
                    ax.scatter([r["Train platos"] for r in supported], [r["F1 %"] for r in supported], alpha=0.6)
                    for row in sorted(supported, key=lambda r: r["L1 aportado g/plato"], reverse=True)[:5]:
                        ax.annotate(row["Clase"], (row["Train platos"], row["F1 %"]), fontsize=9)
                    ax.set(xlabel="Platos de train con la clase", ylabel="F1 validación (%)", title="Soporte vs reconocimiento")
                    ax.grid(alpha=0.2); fig.tight_layout(); plt.show(); plt.close(fig)
            rows = []
            for i in np.argsort(stats["l1"])[::-1][:TOP_DIAGNOSTIC]:
                rows.append({"dish_id": str(arrays["dish_ids"][i]), "Masa real g": diag_fmt(arrays["total_mass"][i]),
                             "Masa estimada g": diag_fmt(arrays["predicted_total_mass"][i]),
                             "Sesgo masa g": diag_fmt(stats["mass_error"][i]), "L1 ingredientes g": diag_fmt(stats["l1"][i]),
                             "L1 con masa real g": diag_fmt(stats["oracle_l1"][i]),
                             "Gramos falsos": diag_fmt(stats["false_mass"][i].sum()),
                             "Ingredientes reales": int(stats["truth"][i].sum()),
                             "FN": int((stats["truth"][i] & ~stats["detected"][i]).sum()) if "detected" in stats else "—",
                             "FP": int((~stats["truth"][i] & stats["detected"][i]).sum()) if "detected" in stats else "—"})
            validation_records = [r for r in records if r["split"] == "validation"]
            index_by_id = {r["dish_id"]: i for i, r in enumerate(validation_records)}
            for row in rows:
                row["PLATE_INDEX (validation)"] = index_by_id.get(row["dish_id"], "—")
            diag_note('Platos con mayor L1: en el punto 10 usar PHOTO_SOURCE="validation" y el PLATE_INDEX mostrado para inspeccionar RGB, crop, depth y atención.')
            diag_table(rows)
            diag_table([{"Cantidad de ingredientes": label, "Platos": int(mask.sum()),
                         "L1 g/plato": diag_fmt(stats["l1"][mask].mean()) if mask.any() else "—",
                         "MAE masa g": diag_fmt(np.abs(stats["mass_error"][mask]).mean()) if mask.any() else "—"}
                        for label, mask in (("1", stats["truth"].sum(axis=1) == 1),
                                            ("2–3", (stats["truth"].sum(axis=1) >= 2) & (stats["truth"].sum(axis=1) <= 3)),
                                            ("4+", stats["truth"].sum(axis=1) >= 4))])
            fig, axes = plt.subplots(1, 2, figsize=(12, 4))
            axes[0].scatter(arrays["total_mass"], arrays["predicted_total_mass"], alpha=0.4)
            bounds = [min(arrays["total_mass"].min(), arrays["predicted_total_mass"].min()),
                      max(arrays["total_mass"].max(), arrays["predicted_total_mass"].max())]
            axes[0].plot(bounds, bounds, "--", color="gray")
            axes[0].set(xlabel="Masa real (g)", ylabel="Masa estimada (g)", title="Sesgo de masa y escala")
            axes[1].scatter(arrays["total_mass"], stats["l1"], alpha=0.4)
            axes[1].set(xlabel="Masa real (g)", ylabel="L1 ingredientes (g)", title="Error según tamaño del plato")
            for ax in axes: ax.grid(alpha=0.2)
            fig.tight_layout(); plt.show(); plt.close(fig)
            diag_title("11.6 Nutrientes: modelo vs perfiles nutricionales")
            try:
                diag_table(diag_display_classes(diag_nutrition_rows(arrays, stats, vocabulary)))
                diag_note("Gramos reales + perfiles del vocabulario muestra discrepancias aun con cantidades correctas: "
                          "promedios por ingrediente, other y anotaciones pueden influir. No es un mínimo de error garantizado: puede haber compensación. "
                          "El NPZ guarda claves pero no hash de perfiles: verificar que vocabulary.json sea el usado al evaluar.")
            except ValueError as exc:
                diag_note(str(exc))
        except (OSError, ValueError, KeyError, IndexError) as exc:
            diag_note(f"No se pudo completar el diagnóstico por plato: {exc}")
    else:
        diag_note("Falta validation/predictions.npz: el punto 9 lo genera al evaluar. No se ejecuta evaluación automáticamente.")
    folder = globals().get("PREDICTIONS")
    photo = globals().get("PHOTO")
    if folder is not None and photo is not None:
        try:
            diag_inference_tensors(Path(folder) / Path(photo).stem)
        except (OSError, ValueError, KeyError, IndexError) as exc:
            diag_note(f"No se pudieron leer los tensores del ejemplo: {exc}")
    else:
        diag_title("11.7 Atención y profundidad")
        diag_note("El punto 10 no está seleccionado; sin tensores de ejemplo. No hace falta ejecutarlo para el diagnóstico global.")
    diag_title("11.8 Qué módulo investigar y qué falta medir")
    diag_table([{"Señal": "Poner masa real reduce mucho el L1", "Investigar": "Cantidad condicional y señales de escala", "Confirmación": "Sesgo por tamaño/perspectiva; distribución de masas"},
                {"Señal": "L1 sigue alto con masa real", "Investigar": "Reconocimiento y proporciones", "Confirmación": "Clases con FN/FP, gramos falsos y grupos de presencia correcta"},
                {"Señal": "Presencias correctas, reparto incorrecto", "Investigar": "Cabezal de cantidad y supervisión de proporciones", "Confirmación": "MAE presente y reparto por clase; atención todavía puede influir"},
                {"Señal": "Gramos correctos, nutrientes incorrectos", "Investigar": "Perfiles nutricionales / other / anotaciones", "Confirmación": "Escenario gramos reales + perfiles; revisar discrepancias"},
                {"Señal": "Train baja; validación empeora sostenidamente", "Investigar": "Generalización, datos, augmentations y regularización", "Confirmación": "Tendencias en varias épocas, no un solo pico"},
                {"Señal": "Beneficio de atención/depth desconocido", "Investigar": "Ablaciones de fusión y atención", "Confirmación": "Entrenar variantes controladas; apagar una rama solo en inferencia no equivale"}])
    diag_note("PRIORIDAD: identificar primero clases que dominan el L1 y gramos falsos; luego comprobar si el reparto falla "
              "aun en platos bien reconocidos. Cambiar un componente por experimento, manteniendo split y presupuesto; "
              "usar validación para elegir y test solo al final. Las fotos personales son controles externos, no sustituyen validación.")
    diag_note("NO DISPONIBLE en estos reportes: gradientes por módulo/pérdida, curvas de componentes en todas las épocas, "
              "calidad espacial de atención con máscaras, exactitud de profundidad en cm, auditoría de escaneos relacionados "
              "y resultado de ablaciones. Requieren instrumentación/anotaciones o experimentos aparte; no se infieren del dashboard.")

run_v3_diagnostics()


## 12. Baseline opcional: V2 + BCE con los MISMOS datos agrupados

No sobrescribe V2 histórica: guarda en v3/baselines/v2_bce_grouped/.
Permite una comparación de modelos con el mismo split, vocabulario y perfiles.
Requiere restaurar el cache V2 224×224 en /content/v3_baselines/, independiente de cache V3.
No se usa ningún checkpoint V2 anterior: el baseline empieza desde ImageNet y cabezales nuevos.


In [ ]:
RUN_V2_BASELINE = False
BASELINE_CONFIG = PROJECT / "config/query2label_100_v2_bce_grouped.json"
BASELINE_RUN = V3 / "baselines/v2_bce_grouped/runs"
BASELINE_DEPTH = Path("/content/v3_baselines/depth_v2_vits")
if RUN_V2_BASELINE:
    if not (BASELINE_DEPTH / "manifest.json").is_file():
        archive = V2 / "depth_v2_vits.zip"
        assert archive.is_file(), f"Falta cache 224 de V2: {archive}"
        local = Path("/content/baseline_depth_224.zip")
        shutil.copy2(archive, local)
        extract_zip_safe(local, BASELINE_DEPTH.parent)
    assert (BASELINE_DEPTH / "manifest.json").is_file(), "Revisar raíz interna del ZIP de cache V2."
    assert not BASELINE_RUN.exists() or not any(BASELINE_RUN.iterdir()), "Elegir otra carpeta para un baseline nuevo."
    run_stream([sys.executable, "-u", "scripts/train_image_model.py", "--config", BASELINE_CONFIG,
                "--dataset-root", DATA, "--processed-dir", PROCESSED, "--depth-cache", BASELINE_DEPTH,
                "--output-dir", BASELINE_RUN, "--torch-cache", DRIVE / "torch_cache",
                "--device", DEVICE, "--workers", "2", "--epochs", "100"])
else:
    print("Baseline V2+BCE agrupado desactivado. No se toca V2 histórica.")


## 13. Comparar reportes guardados — sin entrenar

Por defecto compara V2 histórica vs V3: el informe advierte que no es una comparación controlada,
porque cambian splits y seis clases/perfiles. Para aislar mejor el cambio de modelo, evaluar el baseline
del punto 12 sobre estos mismos datos y apuntar LEFT_REPORT a su reporte.
No decidir experimentos intermedios mirando test.


In [ ]:
RUN_COMPARISON = False
LEFT_REPORT = V2 / "reports/query2label_asl_test"
RIGHT_REPORT = TEST
COMPARISON = V3 / "comparisons" / f"historical_v2_vs_v3_{REPORT_TAG}"
if RUN_COMPARISON:
    run_stream([sys.executable, "-u", "scripts/compare_image_experiments.py", "--left", LEFT_REPORT,
                "--right", RIGHT_REPORT, "--output-dir", COMPARISON])
    display(HTML((COMPARISON / "comparison.html").read_text()))
else:
    print("Comparación desactivada.")


## Qué conservar

Git: código, config V3, config del baseline agrupado, requirements-v3.txt, pruebas, documentos y este notebook **sin salidas**.
Drive: nutrition5k_rgb.zip existente; subir el ZIP de etiquetas a v3/data/. Guardar v3/weights/, depth_v3_320.zip, runs/, reports/, predictions/ y comparisons/.
V2: no borrar ni reemplazar v2/runs/, reports/, predictions/, weights/ o cache; conservar sus best.pt, last.pt y calibrated.pt.
V3 requiere entrenamiento nuevo. La media train se recalcula (provisional actual: 216,89 g); ingredientes mantienen 300 g.
Los pesos iniciales de las nuevas pérdidas son hipótesis de arranque, no resultados optimizados.
